In [1]:
import sys
import scanpy as sc
import anndata
import pandas as pd
import numpy as np
import os

data_type = 'float32'

import matplotlib as mpl
from matplotlib import rcParams
import matplotlib.pyplot as plt
import seaborn as sns

# silence scanpy that prints a lot of warnings
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GridSearchCV, KFold, train_test_split, ParameterGrid
from sklearn.metrics import make_scorer, accuracy_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
import pickle

import tqdm
from tqdm import tqdm

import xgboost
from xgboost import XGBClassifier


merged_features = pd.read_csv('../TPC_data.csv', index_col=0)

In [2]:
merged_features['age'].replace('21yr', 0, inplace=True)
merged_features['age'].replace('22yr', 0, inplace=True)
merged_features['age'].replace('30yr', 1, inplace=True)
merged_features['age'].replace('32yr', 1, inplace=True)
merged_features['age'].replace('33yr', 1, inplace=True)
merged_features['age'].replace('38yr', 1, inplace=True)
merged_features['age'].replace('40yr', 2, inplace=True)
merged_features['age'].replace('42yr', 2, inplace=True)
merged_features['age'].replace('43yr', 2, inplace=True)
merged_features['age'].replace('44yr', 2, inplace=True)
merged_features['age'].replace('46yr', 2, inplace=True)
merged_features['age'].replace('48yr', 2, inplace=True)
merged_features['age'].replace('52yr', 3, inplace=True)
merged_features['age'].replace('53yr', 3, inplace=True)
merged_features['age'].replace('55yr', 3, inplace=True)
merged_features['age'].replace('57yr', 3, inplace=True)
merged_features['age'].replace('59yr', 3, inplace=True)
merged_features['age'].replace('62yr', 4, inplace=True)
merged_features['age'].replace('64yr', 4, inplace=True)
merged_features['age'].replace('66yr', 4, inplace=True)
merged_features['age'].replace('67yr', 4, inplace=True)
merged_features['age'].replace('69yr', 4, inplace=True)

In [3]:
y = merged_features['age']

features = list(merged_features.columns)
features.remove('age')
features.remove('BMI')
X = merged_features[features]

# mutual_info

In [4]:
from sklearn.feature_selection import mutual_info_classif
X_mutual_info = mutual_info_classif(X, y, discrete_features= False)
X_mutual_info=list(X_mutual_info)
X_mutual_info

[0.0,
 0.0,
 0.016151322570979065,
 0.00021400566909779428,
 0.009457603999160735,
 0.0040811366210680156,
 0.0071288117234238335,
 0.0,
 0.003944191589385859,
 0.005180243987538802,
 0.002197955870444801,
 0.00669997790497856,
 0.0,
 0.0037122296748819394,
 0.012083018547999647,
 0.0,
 0.008025608598640765,
 0.0,
 0.0019948751020892885,
 0.009590239784536436,
 0.0001828793836224918,
 0.005222483037502723,
 0.005819580591444051,
 0.0063090571049131405,
 0.0,
 0.0,
 0.0016671976910673791,
 0.0,
 0.012100227124277385,
 0.009801367131852867,
 0.016157037270663643,
 0.004264189452626965,
 0.00282201779291702,
 0.011508764832734553,
 0.005644607356170006,
 0.006437485580632618,
 0.002529191765932026,
 0.0025052750271501445,
 0.003501669545379116,
 0.008848718784066012,
 0.0006399979023345992,
 0.0009481316823078956,
 0.0018953650080559825,
 0.0,
 0.0,
 0.0051167507847798,
 0.0018054167363108498,
 0.0,
 0.006436285905041306,
 0.006534272846129063,
 0.007852106500546352,
 0.000162623264645223

In [5]:
result = [list(t) for t in zip(X_mutual_info, features)]
result = pd.DataFrame(result)
result = result.loc[result[0] > 0.008, :]
features = result[1]
features

2          ISG15
4        TNFRSF4
14          RBP7
16      TNFRSF1B
19         HSPB7
          ...   
2599         NRK
2623       HMGB3
2624       FATE1
2632     MT-ATP8
2633     MT-ND4L
Name: 1, Length: 500, dtype: object

In [6]:
X = merged_features[features]
X.head()

,ISG15,TNFRSF4,RBP7,TNFRSF1B,HSPB7,TEX46,KDM1A,ID3,FAM110D,FCN3,...,SAT1,TIMP1,TSPYL2,CXorf65,ITM2A,NRK,HMGB3,FATE1,MT-ATP8,MT-ND4L
20149X1_AAACGAACACAAATAG-1,-0.504933,3.572244,1.529704,1.953217,-0.274762,-0.524186,-0.539693,1.412673,1.004996,-0.286519,...,0.159355,-0.952402,-0.496833,-0.404458,0.398760,-0.257892,-0.315425,1.542139,-0.503194,0.259315
20149X1_AAACGAACAGACCTAT-1,-0.504933,-0.163379,1.206803,2.458509,-0.274762,0.671765,-0.539693,1.720588,2.849274,-0.286519,...,1.190641,-0.171034,-0.496833,-0.404458,2.247609,-0.257892,-0.315425,-0.263663,-0.503194,0.521004
20149X1_AAACGCTGTAGGACCA-1,1.843142,-0.163379,2.427735,-0.419188,-0.274762,-0.524186,-0.539693,2.008196,1.517562,-0.286519,...,0.959443,0.780182,-0.496833,-0.404458,2.607996,-0.257892,-0.315425,-0.263663,-0.503194,1.228002
20149X1_AAAGAACCACGTTGGC-1,-0.504933,-0.163379,2.121091,-0.419188,-0.274762,-0.524186,-0.539693,2.894780,3.695195,-0.286519,...,1.299590,0.123595,-0.496833,-0.404458,-0.569285,-0.257892,-0.315425,-0.263663,-0.503194,1.606685
20149X1_AAAGTCCGTGGGATTG-1,-0.504933,-0.163379,-0.466696,-0.419188,-0.274762,1.250135,-0.539693,1.497095,-0.385690,2.568544,...,-0.618291,-0.952402,-0.496833,-0.404458,-0.569285,-0.257892,-0.315425,-0.263663,-0.503194,1.088745


# VIF

In [7]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [8]:
vif = [variance_inflation_factor(X.values, X.columns.get_loc(i)) for i in X.columns]
vif

[1.5651729406234194,
 1.2293437993984593,
 4.073589242588674,
 1.4725408550973167,
 1.801967403119422,
 3.0812359909716123,
 1.1371862206974461,
 3.2895387223878942,
 2.3490656119421716,
 1.8044376571526366,
 3.3144961254235152,
 1.0605780466088668,
 3.099532505251465,
 1.409868510181451,
 1.0754116102213858,
 1.2953179368984802,
 1.4876564467643418,
 1.475085684539324,
 1.9196267719289442,
 1.783929885877731,
 3.6446326493524395,
 1.6407801865156384,
 1.523753514103823,
 1.8791753175737895,
 1.3460226996399123,
 3.0557896584842554,
 2.3840386134754077,
 8.691680109098508,
 1.7425457523808296,
 1.1583475703312014,
 2.8115066390432766,
 1.665011652801968,
 3.3187496405545867,
 3.2771618679240744,
 4.218549392304174,
 3.4413489173237988,
 5.569076357124682,
 1.684264216055343,
 2.3655333862634698,
 1.6084452539109515,
 1.8704041962992253,
 1.0583933770935305,
 1.1323334049634113,
 1.2637682777786896,
 1.3440133883165568,
 6.436289881798811,
 1.1385304665471785,
 3.64524361328947,
 2.1883

In [9]:
result_vif = [list(t) for t in zip(vif, features)]
result_vif = pd.DataFrame(result_vif)
result_vif = result_vif.loc[result_vif[0] > 5, :]
result_vif

,0,1
27,8.691680,TXNIP
36,5.569076,ACKR1
45,6.436290,CD34
62,10.459642,TMSB10
118,6.398747,IGFBP7
121,27.381922,SPARCL1
142,8.105396,CD74
161,16.829863,HLA-E
163,15.294249,HLA-B
169,5.753318,HLA-DRA


In [10]:
X = X.drop(columns = result_vif[1])
features = list(X.columns)
X.head()

,ISG15,TNFRSF4,RBP7,TNFRSF1B,HSPB7,TEX46,KDM1A,ID3,FAM110D,FCN3,...,SAT1,TIMP1,TSPYL2,CXorf65,ITM2A,NRK,HMGB3,FATE1,MT-ATP8,MT-ND4L
20149X1_AAACGAACACAAATAG-1,-0.504933,3.572244,1.529704,1.953217,-0.274762,-0.524186,-0.539693,1.412673,1.004996,-0.286519,...,0.159355,-0.952402,-0.496833,-0.404458,0.398760,-0.257892,-0.315425,1.542139,-0.503194,0.259315
20149X1_AAACGAACAGACCTAT-1,-0.504933,-0.163379,1.206803,2.458509,-0.274762,0.671765,-0.539693,1.720588,2.849274,-0.286519,...,1.190641,-0.171034,-0.496833,-0.404458,2.247609,-0.257892,-0.315425,-0.263663,-0.503194,0.521004
20149X1_AAACGCTGTAGGACCA-1,1.843142,-0.163379,2.427735,-0.419188,-0.274762,-0.524186,-0.539693,2.008196,1.517562,-0.286519,...,0.959443,0.780182,-0.496833,-0.404458,2.607996,-0.257892,-0.315425,-0.263663,-0.503194,1.228002
20149X1_AAAGAACCACGTTGGC-1,-0.504933,-0.163379,2.121091,-0.419188,-0.274762,-0.524186,-0.539693,2.894780,3.695195,-0.286519,...,1.299590,0.123595,-0.496833,-0.404458,-0.569285,-0.257892,-0.315425,-0.263663,-0.503194,1.606685
20149X1_AAAGTCCGTGGGATTG-1,-0.504933,-0.163379,-0.466696,-0.419188,-0.274762,1.250135,-0.539693,1.497095,-0.385690,2.568544,...,-0.618291,-0.952402,-0.496833,-0.404458,-0.569285,-0.257892,-0.315425,-0.263663,-0.503194,1.088745


In [11]:
vif = [variance_inflation_factor(X.values, X.columns.get_loc(i)) for i in X.columns]
result_vif = [list(t) for t in zip(vif, features)]
result_vif = pd.DataFrame(result_vif)
result_vif = result_vif.loc[result_vif[0] > 5, :]
result_vif

,0,1


# merge

In [12]:
merged_features_vif = pd.concat([X,y],axis=1) 
merged_features_vif

,ISG15,TNFRSF4,RBP7,TNFRSF1B,HSPB7,TEX46,KDM1A,ID3,FAM110D,FCN3,...,TIMP1,TSPYL2,CXorf65,ITM2A,NRK,HMGB3,FATE1,MT-ATP8,MT-ND4L,age
20149X1_AAACGAACACAAATAG-1,-0.504933,3.572244,1.529704,1.953217,-0.274762,-0.524186,-0.539693,1.412673,1.004996,-0.286519,...,-0.952402,-0.496833,-0.404458,0.398760,-0.257892,-0.315425,1.542139,-0.503194,0.259315,1
20149X1_AAACGAACAGACCTAT-1,-0.504933,-0.163379,1.206803,2.458509,-0.274762,0.671765,-0.539693,1.720588,2.849274,-0.286519,...,-0.171034,-0.496833,-0.404458,2.247609,-0.257892,-0.315425,-0.263663,-0.503194,0.521004,1
20149X1_AAACGCTGTAGGACCA-1,1.843142,-0.163379,2.427735,-0.419188,-0.274762,-0.524186,-0.539693,2.008196,1.517562,-0.286519,...,0.780182,-0.496833,-0.404458,2.607996,-0.257892,-0.315425,-0.263663,-0.503194,1.228002,1
20149X1_AAAGAACCACGTTGGC-1,-0.504933,-0.163379,2.121091,-0.419188,-0.274762,-0.524186,-0.539693,2.894780,3.695195,-0.286519,...,0.123595,-0.496833,-0.404458,-0.569285,-0.257892,-0.315425,-0.263663,-0.503194,1.606685,1
20149X1_AAAGTCCGTGGGATTG-1,-0.504933,-0.163379,-0.466696,-0.419188,-0.274762,1.250135,-0.539693,1.497095,-0.385690,2.568544,...,-0.952402,-0.496833,-0.404458,-0.569285,-0.257892,-0.315425,-0.263663,-0.503194,1.088745,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Older8.TTTACTGTCCACTTCG.1,-0.504933,-0.163379,1.108551,1.719075,-0.274762,-0.524186,1.188605,2.200040,1.575264,1.863989,...,0.335139,-0.496833,-0.404458,0.286136,-0.257892,-0.315425,-0.263663,-0.503194,-0.275360,4
Older8.TTTAGTCCACTAGGTT.1,-0.504933,4.813121,1.956602,-0.419188,-0.274762,-0.524186,1.093138,-0.527442,-0.385690,-0.286519,...,-0.952402,-0.496833,-0.404458,-0.569285,-0.257892,-0.315425,-0.263663,1.224454,0.076884,4
Older8.TTTATGCGTCCGGTGT.1,2.896645,-0.163379,0.901966,0.694665,-0.274762,-0.524186,-0.539693,2.092498,1.290188,1.551356,...,-0.497553,-0.496833,-0.404458,1.199273,-0.257892,-0.315425,-0.263663,-0.503194,1.048292,4
Older8.TTTCAGTAGAGGGTCT.1,-0.504933,-0.163379,-0.466696,2.780469,-0.274762,-0.524186,-0.539693,-0.527442,-0.385690,-0.286519,...,0.836354,-0.496833,-0.404458,-0.569285,-0.257892,-0.315425,-0.263663,2.233176,-0.969345,4


In [13]:
merged_features_vif.to_csv("../TPC_info_0.008_vif_5.csv")